In [ ]:
!pip install sentence-transformers faiss-cpu pandas tqdm

#*Data Loading and Cleaning*

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import tarfile

tar = tarfile.open("20_newsgroups.tar.gz", "r:gz")
tar.extractall()
tar.close()

print("Dataset extracted")

In [ ]:
import os
import pandas as pd

dataset_path = "20_newsgroups"

documents = []
categories = []
filenames = []

for category in os.listdir(dataset_path):
    category_path = os.path.join(dataset_path, category)

    if os.path.isdir(category_path):

        for file in os.listdir(category_path):

            file_path = os.path.join(category_path, file)

            with open(file_path, encoding="latin1") as f:

                text = f.read()

                documents.append(text)
                categories.append(category)
                filenames.append(file)

corpus_df = pd.DataFrame({
    "text": documents,
    "category": categories,
    "file": filenames
})

print("Documents loaded:", len(corpus_df))
corpus_df.head()

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"From:.*\n", "", text)
    text = re.sub(r">.*\n", "", text)
    text = re.sub(r"http\S+", "", text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

corpus_df["clean_text"] = corpus_df["text"].apply(clean_text)

In [ ]:
corpus_df = corpus_df[corpus_df["clean_text"].str.len() > 50]

print("Remaining docs:", len(corpus_df))

In [ ]:
corpus_df.head()

In [ ]:
def remove_usenet_metadata(text):

    # Split header and body using the first blank line
    parts = text.split("\n\n", 1)

    if len(parts) > 1:
        return parts[1]

    return text

In [ ]:
corpus_df["clean_text"] = corpus_df["text"].apply(remove_usenet_metadata)

corpus_df.head()

#Tokenization

In [ ]:
!pip install nltk

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def tokenize_text(text):

    tokens = word_tokenize(text.lower())

    # remove stopwords and very short tokens
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]

    return tokens

In [ ]:
corpus_df["tokens"] = corpus_df["clean_text"].apply(tokenize_text)

print("Tokenization complete")
corpus_df.head()

In [ ]:
print(corpus_df.columns)

In [ ]:
corpus_df["processed_text"] = corpus_df["tokens"].apply(lambda x: " ".join(x))

corpus_df.head()

In [ ]:
corpus_df = corpus_df[corpus_df["processed_text"].str.len() > 30]

print("Remaining documents:", len(corpus_df))

#Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
corpus_df.to_csv("clean_newsgroups_dataset.csv", index=False)

print("Clean dataset saved")

In [ ]:
texts = corpus_df["processed_text"].tolist()

print("Total documents:", len(texts))

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)

In [ ]:
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

In [ ]:
!pip install chromadb

In [ ]:
import numpy as np
import chromadb

In [ ]:
np.save("document_embeddings.npy", embeddings)

print("Embeddings saved")

In [ ]:
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="newsgroups_collection"
)

In [ ]:
batch_size = 4000

In [ ]:
documents = corpus_df["processed_text"].tolist()
categories = corpus_df["category"].tolist()

for i in range(0, len(documents), batch_size):

    batch_docs = documents[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]
    batch_metadata = [{"category": c} for c in categories[i:i+batch_size]]
    batch_ids = [str(j) for j in range(i, i + len(batch_docs))]

    collection.add(
        documents=batch_docs,
        embeddings=batch_embeddings.tolist(),
        metadatas=batch_metadata,
        ids=batch_ids
    )

    print(f"Inserted batch {i} → {i + len(batch_docs)}")

In [ ]:
print(collection.count())

In [ ]:
query = "space shuttle launch nasa"

query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

results["documents"]

In [ ]:
import chromadb

# Initialize a persistent client, specifying a directory to store the database
# If the directory doesn't exist, ChromaDB will create it.
# Make sure to choose a path where you want your database to be saved.
persistent_client = chromadb.PersistentClient(path="./my_chroma_db")

# You can then get or create a collection just like with an in-memory client
persistent_collection = persistent_client.get_or_create_collection(
    name="my_persistent_collection"
)

# Now, any data added to 'persistent_collection' will be saved to './my_chroma_db'
# For example, let's add a document:
persistent_collection.add(
    documents=["This is a test document for persistent storage."],
    metadatas=[{"source": "test"}],
    ids=["doc1"]
)

print("Document added to persistent collection. Data saved to './my_chroma_db' directory.")
print(f"Collection count: {persistent_collection.count()}")

In [ ]:
!zip -r chroma_db.zip my_chroma_db

In [ ]:
from google.colab import files
files.download("chroma_db.zip")

#Clustering

In [ ]:
!pip install umap-learn scikit-learn joblib

In [ ]:
import numpy as np
import umap
import joblib

from sklearn.mixture import GaussianMixture

In [ ]:
print("Embedding shape:", embeddings.shape)

#UMAP Dimensionality Reduction

In [ ]:
reducer = umap.UMAP(
    n_components=20,      # reduce 384 → 20 dimensions
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

reduced_embeddings = reducer.fit_transform(embeddings)

print("Reduced shape:", reduced_embeddings.shape)

#Train GMM Model

In [ ]:
num_clusters = 15

gmm = GaussianMixture(
    n_components=num_clusters,
    covariance_type="full",
    random_state=42
)

gmm.fit(reduced_embeddings)

print("GMM training completed")

In [ ]:
cluster_probs = gmm.predict_proba(reduced_embeddings)

print("Cluster probability matrix:", cluster_probs.shape)

In [ ]:
corpus_df["dominant_cluster"] = cluster_probs.argmax(axis=1)

corpus_df.head()

In [ ]:
np.save("cluster_probabilities.npy", cluster_probs)

print("Cluster probabilities saved")

In [ ]:
corpus_df.to_csv("clustered_documents.csv", index=False)

print("Clustered dataset saved")

In [ ]:
joblib.dump(reducer, "umap_model.pkl")
joblib.dump(gmm, "gmm_model.pkl")

print("Models saved")

##Test Query

In [ ]:
query = "existence of god religion beliefs christian bible discussion"

query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

results["documents"][0]
print(results["metadatas"][0])